# Phân loại review Olist bằng embedding

Khoảng 41% đơn Olist có review kèm lời bình bằng tiếng Bồ Đào Nha. Bài toán: từ **nội dung lời
bình**, dự đoán review có xấu không (`review_score <= 2`). Bạn tự viết tokenizer, vocabulary, một
model "túi embedding" (trung bình embedding các từ rồi qua một tầng Linear), so nó với baseline
TF-IDF + logistic regression, và xem embedding học được gì.

Dùng định nghĩa chung của track: đơn giao xong (`delivered`, có ngày giao), giữ review có
`review_answer_timestamp` mới nhất mỗi đơn, chia theo ngày mua (train trước 2018-01-01, validation
2018-01 tới 2018-03, test 2018-04 tới 2018-08).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Đây là bản lời giải mẫu.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật và train; test bỏ qua.
- Chấm: `uv run pytest tracks/data-science/modules/13-deep-learning/exercises/intermediate/03-olist-review-embeddings -q`

In [ ]:
import collections
import copy
import re
import unicodedata
from pathlib import Path

import pandas as pd
import torch
from torch import nn

TRAIN_END = pd.Timestamp("2018-01-01")
VAL_END = pd.Timestamp("2018-04-01")
TEST_END = pd.Timestamp("2018-09-01")

## 0. Dữ liệu (đã viết sẵn)

`load_reviews` trả một dòng mỗi đơn giao xong có lời bình không rỗng: `order_id`,
`order_purchase_timestamp`, `text`, `review_score`, `bad`.

In [ ]:
def load_reviews(raw_dir) -> pd.DataFrame:
    raw = Path(raw_dir)
    dates = ["order_purchase_timestamp", "order_delivered_customer_date"]
    orders = pd.read_csv(raw / "olist_orders_dataset.csv", parse_dates=dates)
    reviews = pd.read_csv(raw / "olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
    df = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
    latest = reviews.sort_values("review_answer_timestamp", kind="stable").drop_duplicates("order_id", keep="last")
    df = df.merge(latest[["order_id", "review_score", "review_comment_message"]], on="order_id")
    df["text"] = df.review_comment_message.fillna("").str.strip()
    df = df[df.text.str.len() > 0].copy()
    df["bad"] = (df.review_score <= 2).astype(int)
    return df[["order_id", "order_purchase_timestamp", "text", "review_score", "bad"]].reset_index(drop=True)


def time_split(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    t = df.order_purchase_timestamp
    return {"train": df[t < TRAIN_END], "val": df[(t >= TRAIN_END) & (t < VAL_END)],
            "test": df[(t >= VAL_END) & (t < TEST_END)]}

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
reviews = load_reviews(ROOT / "data" / "raw")
splits = time_split(reviews)
print(len(reviews), "review có lời bình; tỉ lệ xấu", round(reviews.bad.mean(), 4))
print({k: (len(v), round(v.bad.mean(), 4)) for k, v in splits.items()})
print(reviews.text.str.split().str.len().describe().round(1).to_dict())
reviews.sample(5, random_state=0)[["review_score", "text"]]

## 1. Tokenizer và vocabulary

- `tokenize(text)`: chữ thường, **bỏ dấu** (người Brazil hay gõ "nao" thay cho "não"; chuẩn hoá
  `unicodedata.normalize("NFKD", ...)` rồi bỏ ký tự có `unicodedata.combining(c)`), giữ các cụm
  `[a-z]+` (bỏ số, dấu câu, emoji).
- `build_vocab(texts, min_freq=2, max_size=None)`: `{"<pad>": 0, "<unk>": 1}` rồi các từ có tần
  suất ≥ `min_freq`, sắp theo tần suất giảm dần, hoà thì theo thứ tự chữ cái. `max_size` tính cả
  hai token đặc biệt.
- `encode(texts, vocab, max_len)`: `LongTensor (N, max_len)`, cắt bớt nếu dài, đệm 0 nếu ngắn, từ
  lạ thành 1.

In [ ]:
def tokenize(text: str) -> list[str]:
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.findall(r"[a-z]+", text)


def build_vocab(texts, min_freq: int = 2, max_size: int | None = None) -> dict[str, int]:
    counts = collections.Counter(tok for text in texts for tok in tokenize(text))
    words = sorted((w for w, c in counts.items() if c >= min_freq), key=lambda w: (-counts[w], w))
    if max_size is not None:
        words = words[:max_size - 2]
    vocab = {"<pad>": 0, "<unk>": 1}
    for w in words:
        vocab[w] = len(vocab)
    return vocab


def encode(texts, vocab: dict[str, int], max_len: int = 50) -> torch.Tensor:
    out = torch.zeros(len(texts), max_len, dtype=torch.long)
    for i, text in enumerate(texts):
        ids = [vocab.get(tok, 1) for tok in tokenize(text)][:max_len]
        if ids:
            out[i, :len(ids)] = torch.tensor(ids)
    return out

In [ ]:
# explore: kích thước vocabulary và tỉ lệ từ lạ trên validation, test
vocab = build_vocab(splits["train"].text, min_freq=2)
print("vocab", len(vocab))
for name in ("val", "test"):
    toks = [t for text in splits[name].text for t in tokenize(text)]
    print(name, "tỉ lệ <unk>", round(sum(t not in vocab for t in toks) / len(toks), 4))
X_tr, X_va, X_te = (encode(splits[k].text.tolist(), vocab, max_len=50) for k in ("train", "val", "test"))
y_tr, y_va, y_te = (torch.tensor(splits[k].bad.to_numpy(), dtype=torch.float32) for k in ("train", "val", "test"))

## 2. Model túi embedding

`BagOfEmbeddings(vocab_size, dim)` có đúng hai thuộc tính tham số:

- `self.embedding`: `nn.Embedding` hoặc `nn.EmbeddingBag` (`vocab_size × dim`);
- `self.out`: `nn.Linear(dim, 1)`.

`forward(ids)` với `ids (N, L)` trả logits `(N,)`: **trung bình embedding của các token khác 0**
(dòng toàn padding cho vector 0), rồi qua `self.out`. Padding không được ảnh hưởng tới kết quả.

In [ ]:
class BagOfEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, dim: int = 32):
        super().__init__()
        # EmbeddingBag gộp lookup + mean trong một kernel, bỏ qua padding_idx khi tính trung bình
        self.embedding = nn.EmbeddingBag(vocab_size, dim, mode="mean", padding_idx=0)
        self.out = nn.Linear(dim, 1)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        return self.out(self.embedding(ids)).squeeze(-1)

## 3. Train có early stopping

`train_text_model(model, X, y, X_val, y_val, epochs=10, lr=1e-2, batch_size=256, patience=2, seed=0)`:
Adam, `BCEWithLogitsLoss`, xáo bằng `torch.Generator().manual_seed(seed + epoch)`, sau mỗi epoch đo
val loss (chế độ eval, không gradient), early stopping và khôi phục trọng số tốt nhất **giống hệt
quy tắc của bài Basic 02**. Trả `{"train_loss": [...], "val_loss": [...], "best_epoch": int}`.

In [ ]:
def train_text_model(model: nn.Module, X, y, X_val, y_val, epochs: int = 10, lr: float = 1e-2,
                     batch_size: int = 256, patience: int = 2, seed: int = 0) -> dict:
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    hist = {"train_loss": [], "val_loss": [], "best_epoch": -1}
    best, best_state, since = float("inf"), None, 0
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X), generator=torch.Generator().manual_seed(seed + epoch))
        total = 0.0
        for start in range(0, len(X), batch_size):
            idx = order[start:start + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(X[idx]), y[idx])
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        hist["train_loss"].append(total / len(X))
        model.eval()
        with torch.no_grad():
            val = loss_fn(model(X_val), y_val).item()
        hist["val_loss"].append(val)
        if val < best:
            best, since, hist["best_epoch"] = val, 0, epoch
            best_state = copy.deepcopy(model.state_dict())
        else:
            since += 1
            if since >= patience:
                break
    model.load_state_dict(best_state)
    return hist

In [ ]:
# explore: baseline TF-IDF + logistic regression (cùng tokenizer) và model túi embedding
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

tfidf = TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None, min_df=2)
A_tr = tfidf.fit_transform(splits["train"].text)
logreg = LogisticRegression(C=4, max_iter=2000).fit(A_tr, splits["train"].bad)
rows = {}
for name in ("val", "test"):
    p = logreg.predict_proba(tfidf.transform(splits[name].text))[:, 1]
    rows[("TF-IDF + LR", name)] = (roc_auc_score(splits[name].bad, p), average_precision_score(splits[name].bad, p))

torch.manual_seed(0)
bag = BagOfEmbeddings(len(vocab), dim=32)
start = time.perf_counter()
hist = train_text_model(bag, X_tr, y_tr, X_va, y_va, epochs=10, lr=1e-2, batch_size=256, patience=2, seed=0)
print("túi embedding: best_epoch", hist["best_epoch"], "val_loss", [round(v, 4) for v in hist["val_loss"]],
      f"{time.perf_counter() - start:.1f} s")
bag.eval()
with torch.no_grad():
    for name, X_, y_ in (("val", X_va, y_va), ("test", X_te, y_te)):
        p = torch.sigmoid(bag(X_)).numpy()
        rows[("Túi embedding", name)] = (roc_auc_score(y_, p), average_precision_score(y_, p))
pd.DataFrame(rows, index=["AUC", "PR-AUC"]).T.round(4)

## 4. Embedding học được gì?

`nearest_words(weight, vocab, word, k=5)`: k từ có **cosine similarity** cao nhất với `word` trong
ma trận `weight (V, dim)`, bỏ chính nó, `<pad>` và `<unk>`; từ không có trong vocab thì `KeyError`.

In [ ]:
def nearest_words(weight: torch.Tensor, vocab: dict[str, int], word: str, k: int = 5) -> list[str]:
    idx = vocab[word]                                   # KeyError nếu không có
    w = torch.nn.functional.normalize(weight.detach().float(), dim=1)
    sims = w @ w[idx]
    sims[[0, 1, idx]] = float("-inf")                   # bỏ <pad>, <unk>, chính nó
    inverse = {i: t for t, i in vocab.items()}
    return [inverse[i] for i in sims.topk(k).indices.tolist()]

In [ ]:
# explore: hàng xóm gần nhất trong embedding của model phân loại, và các từ "kéo về xấu" mạnh nhất
W = bag.embedding.weight.detach()
for w in ["pessimo", "otimo", "atraso", "defeito"]:
    print(w, nearest_words(W, vocab, w))
score = (W @ bag.out.weight.detach().T).squeeze()      # đóng góp của từng từ vào logit
inverse = {i: t for t, i in vocab.items()}
print("kéo về xấu:", [inverse[i] for i in score.topk(10).indices.tolist()])
print("kéo về tốt:", [inverse[i] for i in (-score).topk(10).indices.tolist()])

## 5. Trả lời

1. Không: TF-IDF + LR cao hơn một chút cả AUC lẫn PR-AUC. Với khoảng 18 nghìn review train, ngắn
   (trung vị 9 từ), tín hiệu chủ yếu là từ khoá ("pessimo", "nao recebi"), mô hình tuyến tính trên
   túi từ đã gần trần. Túi embedding cũng là mô hình tuyến tính theo trung bình các từ, chỉ khác là
   phải học thêm embedding từ đầu với ít dữ liệu. Deep learning cho văn bản thắng rõ khi có nhiều dữ
   liệu, khi thứ tự từ quan trọng, hoặc khi dùng embedding/model đã pretrain.
2. Không hợp lý: hàng xóm gần nhất gần như ngẫu nhiên. Model chỉ cần một hướng trong không gian
   embedding (hướng mà `out` nhân vào) để phân loại; các chiều còn lại không bị ràng buộc gì nên
   không mang nghĩa. Hướng "kéo về xấu/tốt" chính là thứ model được train để học nên rõ ràng.
   Muốn embedding có nghĩa chung (đồng nghĩa ở gần nhau) cần mục tiêu tự giám sát trên nhiều văn bản
   như word2vec (mục 6 lý thuyết) hoặc model ngôn ngữ pretrain.
3. Khoảng 23.8%, gần gấp đôi 12.8%: khách không vui viết lời bình nhiều hơn. Model chỉ áp dụng được
   cho đơn **có** lời bình, và nhãn gần như nằm trong chính lời bình (chẩn đoán, không phải dự báo).
   Nó hữu ích để phân loại / ưu tiên xử lý review đã viết, không thay được model dự báo trước khi
   khách review (module 11, 12).